# nn

> This module contains lucidlearn's `make_*` functions for assembling deep-learning models. These range from simple building blocks like `make_linear` to specialized architectures like `make_spectrum_encoder`, which embeds peaks from tandem mass spectrometry (MS/MS) experiments so that, given the right model head, molecular properties or structures can be predicted from the spectra.

In [ ]:
#| default_exp nn

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import jax
import jax.numpy as jnp

In [ ]:
#| export
def make_linear(key, fan_in, fan_out, act_fn=jax.nn.relu, initializer=jax.nn.initializers.he_normal, bias=True, act=True):
    init = initializer()
    params = {}
    params["w"] = init(key, (fan_in, fan_out), jnp.float32)
    if bias: params["b"] = jnp.zeros(fan_out)
    
    def lin_fn(params, x):
        x = x@params["w"]
        if bias: x += params["b"]
        if act: x = act_fn(x)
        return x
    
    return params, lin_fn

def make_conv2d(key, cho, chi, ks=(3, 3), padding='SAME', strides=(1, 1), act_fn=jax.nn.relu, initializer=jax.nn.initializers.he_normal, bias=True, act=True):
    init = initializer()
    params = {}
    params["w"] = init(key, (cho, chi, *ks), jnp.float32)
    if bias: params["b"] = jnp.zeros(cho)
    
    def conv2d_fn(params, x):
        x = jax.lax.conv_general_dilated(x, params["w"], padding=padding, window_strides=strides)
        if bias: x += params["b"][None, :, None, None]
        if act: x = act_fn(x)
        return x
    
    return params, conv2d_fn

def make_squeeze(key, dims=None):
    
    def squeeze(params, x): 
        if dims is None:
            return x.squeeze()
        return jnp.squeeze(x, axis=dims)
    
    return {}, squeeze

def make_layernorm1d(key, num_features, eps=1e-5, offset=True):
    params = {}
    params["scale"] = jnp.ones(num_features)
    if offset: params["offset"] = jnp.zeros(num_features)
    
    def layernorm_fn(params, x):
        mean = jnp.mean(x, axis=-1, keepdims=True)
        var = jnp.var(x, axis=-1, keepdims=True)
        x_norm = (x - mean) / jnp.sqrt(var + eps)
        x = x_norm * params["scale"][None, None, :]
        if offset: x += params["offset"][None, None, :]
        return x
    
    return params, layernorm_fn

def make_layernorm2d(key, num_features, eps=1e-5, offset=True, ndims=4):
    broadcast_dims = tuple(None for i in range(ndims-2))
    axes = tuple(range(1, ndims))
    params = {}
    params["scale"] = jnp.ones(num_features)
    if offset: params["offset"] = jnp.zeros(num_features)
    
    def layernorm2d_fn(params, x):
        mean = jnp.mean(x, axis=axes)[:, None, *broadcast_dims]
        var = jnp.var(x, axis=axes)[:, None, *broadcast_dims]
        x_norm = (x - mean) / jnp.sqrt(var + eps)
        x = x_norm * params["scale"][None, :, *broadcast_dims]
        if offset: x += params["offset"][None, :, *broadcast_dims]
        return x
    
    return params, layernorm2d_fn

def make_sinusoidal_embedding(d=512, ldmin=10**(-2.5), ldmax=10**(3.3)):
    i = jnp.arange(d // 2)
    lds = 2 * jnp.pi * 1/(ldmin * (ldmax/ldmin)**((2 * i)/(d-2)))

    def sinusoidal_embedding_fn(params, x):
        sine = jnp.sin(lds[None, :] * x[..., None])
        cosine = jnp.cos(lds[None, :] * x[..., None])
        x = jnp.concat((sine, cosine), axis=-1)
        return x
    
    return {}, sinusoidal_embedding_fn

def make_peak_embedding(key, d=512, hidden_layer=512, ldmin=10**(-2.5), ldmax=10**(3.3), act_fn=jax.nn.relu, initializer=jax.nn.initializers.he_normal, bias=True):
    params = {}
    keys = random.split(key, 4)
    params['Linear_11'], ln11 = make_linear(keys[0], d, hidden_layer, act_fn=act_fn, initializer=initializer, bias=True, act=True)
    params['Linear_12'], ln12 = make_linear(keys[1], hidden_layer, d, act_fn=act_fn, initializer=initializer, bias=True, act=False)
    params['Linear_21'], ln21 = make_linear(keys[2], d+1, hidden_layer, act_fn=act_fn, initializer=initializer, bias=True, act=True)
    params['Linear_22'], ln22 = make_linear(keys[3], hidden_layer, d, act_fn=act_fn, initializer=initializer, bias=True, act=False)
    _, se = make_sinusoidal_embedding(d=d, ldmin=ldmin, ldmax=ldmax)
    
    def peak_embedding_fn(params, x):
        mzs = x[..., 0]
        ints = x[..., 1:2]
        mlp1_in = se({}, mzs)
        res1 = ln12(params['Linear_12'], ln11(params['Linear_11'], mlp1_in))
        mlp2_in = jnp.concat((res1, ints), axis=-1)
        res2 = ln22(params['Linear_22'], ln21(params['Linear_21'], mlp2_in))
        return res2

    return params, peak_embedding_fn

def mask_fn(x): return x[..., 0] != 0.0

def make_multi_head_attention(key, nembd, nhead, attention_backend=None, initializer=jax.nn.initializers.he_normal):
    params = {}
    keys = random.split(key, 4)
    params["Q"], q_fn = make_linear(keys[0], nembd, nembd, initializer=initializer, bias=False, act=False)
    params["K"], k_fn = make_linear(keys[1], nembd, nembd, initializer=initializer,  bias=False, act=False)
    params["V"], v_fn = make_linear(keys[2], nembd, nembd, initializer=initializer,  bias=False, act=False)
    params["O"], o_fn = make_linear(keys[3], nembd, nembd, initializer=initializer,  bias=False, act=False)

    def multi_head_attention_fn(params, x, mask=None):
        B, T, C = x.shape
        H = nembd//nhead
        q = q_fn(params["Q"], x).reshape(B, T, nhead, H) # B, T, C --> B, T, N, H
        k = k_fn(params["K"], x).reshape(B, T, nhead, H) # B, T, C --> B, T, N, H
        v = v_fn(params["V"], x).reshape(B, T, nhead, H) # B, T, C --> B, T, N, H
        o = o_fn(params["O"], jax.nn.dot_product_attention(q, k, v, mask=mask, implementation=attention_backend).reshape(B, T, C))
        return o
    
    return params, multi_head_attention_fn

def make_transformer_block(key, nembd, nhead, ffdim=None, attention_backend=None, act_fn=jax.nn.relu, initializer=jax.nn.initializers.he_normal, bias=True):
    params = {}
    keys = random.split(key, 3)
    ffdim = ffdim if ffdim else 4 * nembd
    params["MultiHeadAttention"], mha_fn = make_multi_head_attention(keys[0], nembd, nhead, attention_backend=attention_backend, initializer=initializer)
    params["LayerNorm_1"], ln1_fn = make_layernorm1d(None, num_features=nembd)
    params["LayerNorm_2"], ln2_fn = make_layernorm1d(None, num_features=nembd)
    params['Linear_1'], ffn1 = make_linear(keys[1], nembd, ffdim, act_fn=act_fn, initializer=initializer, bias=bias, act=True)
    params['Linear_2'], ffn2 = make_linear(keys[2], ffdim, nembd, act_fn=act_fn, initializer=initializer, bias=bias, act=False)

    def transformer_block_fn(params, x, mask=None):
        x = x + mha_fn(params["MultiHeadAttention"], ln1_fn(params["LayerNorm_1"], x), mask=mask)
        x = x + ffn2(params['Linear_2'], ffn1(params['Linear_1'], ln2_fn(params["LayerNorm_2"], x)))
        return x
    
    return params, transformer_block_fn
    
def make_spectrum_encoder(key, d=512, ldmin=10**(-2.5), ldmax=10**(3.3), hidden_layer=512, ffdim=512, nhead=32, nlayers=6, act_fn=jax.nn.relu, initializer=jax.nn.initializers.he_normal, attention_backend=None, mask_fn=mask_fn):
    params = {}
    key, pe_key = random.split(key)
    params["PeakEmbedding"], pe_fn = make_peak_embedding(pe_key, d=d, hidden_layer=hidden_layer, ldmin=ldmin, ldmax=ldmax, act_fn=act_fn, initializer=initializer)
    tb_param_list = []
    for i in range(nlayers):
        key, tb_key = random.split(key)
        p, tb_fn = make_transformer_block(tb_key, d, nhead, ffdim=ffdim, act_fn=act_fn, initializer=initializer, attention_backend=attention_backend)
        tb_param_list.append(p)
    stacked = jax.tree.map(lambda *xs: jnp.stack(xs), *tb_param_list)
    params["TransformerBlocks"] = stacked
    params["LayerNorm"], ln_fn = make_layernorm1d(None, d)

    def spectrum_encoder_fn(params, x):
        mask = mask_fn(x)[:, None, None, :] if mask_fn else None
        def _step(x, layer_params):
            x = tb_fn(layer_params, x, mask)
            return x, None
        x = pe_fn(params["PeakEmbedding"], x)
        x, _ = jax.lax.scan(_step, x, params["TransformerBlocks"])
        x = ln_fn(params["LayerNorm"], x)[:, 0] # <-- slice off only the precursor embedding peak
        return x
    
    return params, spectrum_encoder_fn

def make_property_predictor(key, d=512, ldmin=10**(-2.5), ldmax=10**(3.3), hidden_layer=512, ffdim=512, nhead=32, nlayers=6, dout=10, act_fn=jax.nn.relu, initializer=jax.nn.initializers.he_normal, attention_backend=None, bias=True, mask_fn=mask_fn):
    params = {}
    keys = random.split(key, num=3)
    params["SpectrumEncoder"], spectrum_encoder_fn = make_spectrum_encoder(keys[0], d=d, ldmin=ldmin, ldmax=ldmax, hidden_layer=hidden_layer, ffdim=ffdim, nhead=nhead, nlayers=nlayers, act_fn=act_fn, initializer=initializer, attention_backend=attention_backend, mask_fn=mask_fn)
    params['Linear_1'], ffn1 = make_linear(keys[1], d, hidden_layer, act_fn=act_fn, initializer=initializer, bias=bias, act=True)
    params['Linear_2'], ffn2 = make_linear(keys[2], hidden_layer, dout, act_fn=act_fn, initializer=initializer, bias=bias, act=False)

    def property_predictor_fn(params, x):
        x = spectrum_encoder_fn(params["SpectrumEncoder"], x)
        x = ffn2(params['Linear_2'], ffn1(params['Linear_1'], x))
        return x
    
    return params, property_predictor_fn

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()